# Extract sequence embeddings
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/extract_embeddings.ipynb)

Extract one mean-pooled vector per isolated IDR and save the array with its sequence index.


## Setup
Use a standard Colab runtime, or select **Runtime → Change runtime type → GPU** and update `DEVICE` below.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from idiom import IDiom
from idiom.utils.notebook_helpers import (
    check_context,
    example_file,
    isolated,
    load_inputs,
    save_run,
    summaries,
)

started = time.perf_counter()

## Settings
Upload a FASTA through Colab’s Files pane, or use the example. Annotated protein headers use `_IDR_start-end` with 1-based, inclusive coordinates.


In [2]:
INPUT_FASTA = None # Default: ProtGPS nucleolus IDRs; otherwise set a FASTA path
INPUT_MODE = "idr" # "idr" for isolated IDRs; "annotated" for full proteins with IDR spans
MAX_RECORDS = 32 # Maximum accepted records; None uses all
MODEL_ID = "jxliu2/idiom-20M" # Pretrained IDiom model or local release directory
LAYER = 5 # Zero-based transformer block; must exist in MODEL_ID
DEVICE = "cpu" # "cpu" or "cuda"; "auto" uses an available GPU
OUT_DIR = Path("embedding_outputs") / time.strftime("%Y%m%d-%H%M%S") # New timestamped folder per run

## Load sequences


In [3]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
input_path = INPUT_FASTA or example_file("protgps/nucleolus.fasta", Path("example_inputs"))
records, audit = load_inputs(input_path, INPUT_MODE, MAX_RECORDS)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
if not records:
    raise ValueError("No accepted sequences; review input_audit.csv.")
embedding_records = isolated(records)

## Extract embeddings
Each row is the mean of the IDR residue embeddings. Annotated protein flanks are removed before encoding.


In [4]:
model = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
check_context(embedding_records, model.model.cfg.max_seq_len)
values, _ = model.embed(embedding_records, layers=[LAYER], pool="mean")[LAYER]
print("Embedding shape:", values.shape)

Embedding shape: (32, 512)


## Save
`embeddings.npy` has shape `(number of sequences, model width)`. Row numbers correspond to `embedding_index.csv`.


In [5]:
np.save(OUT_DIR / "embeddings.npy", values)
index = summaries(records, audit)[["record_id", "accession", "sequence"]]
index.to_csv(OUT_DIR / "embedding_index.csv", index_label="embedding_row")

display(index.head(3))
preview = pd.DataFrame(
    values[:3, :6],
    index=index.record_id.iloc[:3],
    columns=[f"dimension_{i}" for i in range(min(6, values.shape[1]))],
)
print("Embedding preview: first three sequences and six dimensions")
display(preview)

,record_id,accession,sequence
0,record_0,protgps_00027,LSSVLDHPFMSRNSSTKSKDLGTVEDSIDSGHATISTAITASSSTS...
1,record_1,protgps_00028,APFFPIIIGRKPGSTSSPKALSPPPSVDSNYPTRERASFNRMVMHS...
2,record_2,protgps_00029,MGRSRRTGAHRAHSLARQMKAKRRRPDLDEIHRELRPQGSARPQPD...


Embedding preview: first three sequences and six dimensions


,dimension_0,dimension_1,dimension_2,dimension_3,dimension_4,dimension_5
record_id,,,,,,
record_0,-9.454997,-3.380577,370.921326,-790.305725,-10.096302,7.187456
record_1,-2.882345,-2.960331,343.723206,-710.632507,-3.238566,5.714835
record_2,-5.986372,-1.781914,398.657043,-818.509338,5.830194,-1.916594


## Variants: last pooling and per-residue extraction
The examples below use only the first two accepted IDRs. `last` returns one vector per sequence from its final IDR residue, excluding EOS; `none` returns a vector for each residue. Keep model, layer, and context conventions fixed when choosing a representation.


In [6]:
examples = embedding_records[:2]
last_values, last_index = model.embed(examples, layers=[LAYER], pool="last")[LAYER]
np.save(OUT_DIR / "last_embeddings.npy", last_values)
pd.DataFrame(last_index).to_csv(OUT_DIR / "last_index.csv", index_label="embedding_row")
print("Last pooling:", last_values.shape)

Last pooling: (2, 512)


Per-residue metadata identifies the input record, residue, and source position. Because these inputs have been isolated, `source_pos` is a **0-based position within the IDR**. Add the original record’s `idr_start` to recover its protein position.


In [7]:
residue_values, residue_index = model.embed(examples, layers=[LAYER], pool="none")[LAYER]
residue_table = pd.DataFrame(residue_index)
residue_table["protein_position_1based"] = [
    row["source_pos"] + records[row["record_idx"]].idr_start + 1 for row in residue_index
]
np.save(OUT_DIR / "residue_embeddings.npy", residue_values)
residue_table.to_csv(OUT_DIR / "residue_index.csv", index_label="embedding_row")
print("Per-residue:", residue_values.shape)
display(residue_table.head(3))

Per-residue: (438, 512)


,record_idx,accession,source_pos,residue,is_idr,protein_position_1based
0,0,record_0,0,L,True,1
1,0,record_0,1,S,True,2
2,0,record_0,2,S,True,3


## Results
`embeddings.npy` and `embedding_index.csv` store mean-pooled vectors and their sequence IDs. `last_embeddings.npy` and `residue_embeddings.npy` store the two alternative representations with matching index CSV files.

`run.json` records the settings and package versions. Open the output folder in Colab’s **Files** pane to download results. Download them before the runtime ends, or copy them to mounted Drive. Saved notebook previews do not include the exported files.


Saved previews show a CPU example run. Run the cells to create the exported files.

In [8]:
save_run(
    OUT_DIR,
    dict(
        model=MODEL_ID,
        device=DEVICE,
        input=str(input_path),
        input_mode=INPUT_MODE,
        max_records=MAX_RECORDS,
        layer=LAYER,
        pool="mean",
        context="isolated IDR",
    ),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: embedding_outputs/20260921-134144
